# QAOA + GNN: Variational Parameter Prediction for MaxCut

This notebook benchmarks two strategies for selecting QAOA variational parameters on small MaxCut instances:

1. **Classical per-instance optimization** via Nelder–Mead on the numpy statevector simulator.
2. **GNN-based prediction** using a pre-trained (or randomly initialized) GCN that maps graph structure to $(\gamma, \beta)$ in a single forward pass.

Results include: optimized expectation values, inference timing, and the expected-cut landscape as a function of $\gamma$.

**QAOA depth:** $p=1$.  **Simulator:** exact statevector over $2^n$ amplitudes.

In [ ]:
import sys, os, math, time

def find_project_root():
    p = os.getcwd()
    while True:
        if os.path.isdir(os.path.join(p, 'src')):
            return p
        parent = os.path.dirname(p)
        if parent == p:
            return os.getcwd()
        p = parent

proj_root = find_project_root()
if proj_root not in sys.path:
    sys.path.insert(0, proj_root)

import numpy as np
import torch
from scipy.optimize import minimize
import matplotlib.pyplot as plt
import networkx as nx

from src.gnn import SimpleGCN
from src.qaoa_sim import qaoa_state, expected_cut
from src.data import sample_erdos_renyi

print('Project root:', proj_root)
print('PyTorch:', torch.__version__)

In [ ]:
# Load or initialize GNN
p = 1
model = SimpleGCN(in_feats=1, hidden=32, out_feats=2, p=p)
model_path = os.path.join(proj_root, 'model.pt')
if os.path.exists(model_path):
    try:
        model.load_state_dict(torch.load(model_path, map_location='cpu'))
        print('Loaded trained model from', model_path)
    except Exception as e:
        print('Model load failed:', e)
else:
    print('model.pt not found — using untrained (random) weights')
model.eval()

In [ ]:
def classical_optimize(n, edges, p, maxiter=300, seed=0):
    """Nelder-Mead minimization of -<C>(γ, β)."""
    rng = np.random.default_rng(seed)
    def neg_exp(x):
        return -expected_cut(n, edges, qaoa_state(n, edges, x[:p], x[p:]))
    x0 = rng.uniform(0, math.pi, 2 * p)
    res = minimize(neg_exp, x0, method='Nelder-Mead',
                   options={'maxiter': maxiter, 'xatol': 1e-5, 'fatol': 1e-5})
    return res.x[:p], res.x[p:]

In [ ]:
# Test graph
n = 6
G = sample_erdos_renyi(n, p_edge=0.5, seed=123)
edges = list(G.edges())
print(f'Graph: n={n}, m={len(edges)}, edges={edges}')

In [ ]:
# Classical optimization
t0 = time.perf_counter()
opt_gammas, opt_betas = classical_optimize(n, edges, p)
classical_time = time.perf_counter() - t0

state_opt = qaoa_state(n, edges, opt_gammas, opt_betas)
val_opt = expected_cut(n, edges, state_opt)
print(f'Classical <C>_opt = {val_opt:.4f}  ({classical_time*1000:.1f} ms)')
print(f'  gamma={opt_gammas}, beta={opt_betas}')

In [ ]:
# GNN prediction
A = nx.to_numpy_array(G) + np.eye(n)
X = A.sum(axis=1, keepdims=True).astype(np.float32)
At = torch.tensor(A, dtype=torch.float32)
Xt = torch.tensor(X, dtype=torch.float32)

with torch.no_grad():
    _ = model(Xt, At)  # warm-up

t0 = time.perf_counter()
with torch.no_grad():
    out = model(Xt, At).view(-1).numpy()
gnn_time = time.perf_counter() - t0

pred_g, pred_b = out[:p], out[p:]
state_pred = qaoa_state(n, edges, pred_g, pred_b)
val_pred = expected_cut(n, edges, state_pred)

print(f'GNN     <C>_pred  = {val_pred:.4f}  ({gnn_time*1e6:.0f} µs)')
print(f'  gamma={pred_g}, beta={pred_b}')
if gnn_time > 0:
    print(f'Speedup (classical/GNN): {classical_time/gnn_time:.0f}×')

In [ ]:
# Visualization: graph topology + expectation landscape
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
pos = nx.spring_layout(G, seed=42)
nx.draw(G, pos, ax=ax, with_labels=True,
        node_color='#4C9BE8', edge_color='#888', node_size=500, font_color='white')
ax.set_title(f'MaxCut test graph (n={n}, m={len(edges)})')

ax = axes[1]
beta_fixed = float(opt_betas[0])
gamma_sweep = np.linspace(0, np.pi, 120)
exp_vals = [expected_cut(n, edges, qaoa_state(n, edges, [g], [beta_fixed]))
            for g in gamma_sweep]
ax.plot(gamma_sweep, exp_vals, lw=2, color='steelblue', label='⟨C⟩(γ)')
ax.axvline(float(opt_gammas[0]), color='red',    ls='--', label=f'γ_opt={opt_gammas[0]:.3f}')
ax.axvline(float(pred_g[0]),     color='orange', ls=':',  label=f'γ_GNN={pred_g[0]:.3f}')
ax.set_xlabel('γ'); ax.set_ylabel('⟨C⟩')
ax.set_title(f'Expectation landscape (β={beta_fixed:.3f})')
ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Visualization explanation: graph + expectation landscape

Left: the MaxCut instance — nodes and edges of the test graph. Right: the expected cut value ⟨C⟩ as a function of the QAOA parameter γ with β held fixed at the classical optimum.

Interpretation: Vertical lines show the classical (red) and GNN-predicted (orange) γ. Peaks in the landscape are promising γ choices; closeness of γ_GNN to γ_opt indicates the GNN's ability to predict high-quality parameters.

In [ ]:
# Timing bar chart
methods  = ['Nelder–Mead\n(classical)', 'GNN\n(inference)']
times_ms = [classical_time * 1e3, gnn_time * 1e3]
colors   = ['#d9534f', '#5cb85c']

fig, ax = plt.subplots(figsize=(5, 4))
bars = ax.bar(methods, times_ms, color=colors, width=0.5, edgecolor='black')
for bar, t in zip(bars, times_ms):
    label = f'{t:.3f} ms' if t >= 0.01 else f'{t*1e3:.1f} µs'
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() * 1.03,
            label, ha='center', va='bottom', fontsize=10)
ax.set_yscale('log')
ax.set_ylabel('Wall-clock time (ms, log scale)')
ax.set_title('Per-instance parameter estimation latency')
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

### Visualization explanation: timing chart

This bar chart compares wall-clock latency for per-instance classical optimization (Nelder–Mead) and a single forward pass of the GNN. The y-axis is log-scaled to show large differences in runtime clearly. A large gap illustrates the GNN's potential for low-latency parameter estimation in real-time applications.

## Technical notes

- **Statevector simulator complexity:** $O(p \cdot 2^n)$ per evaluation; scales exponentially in $n$. Practical upper bound is $n \approx 12$ on a laptop.
- **GNN input features:** augmented degree vector $d_v^{+} = \deg(v) + 1$ (from $\hat{A} = A + I$), which encodes local connectivity while keeping the pipeline simple.
- **Training objective:** minimize $-\mathbb{E}_{G}[\langle C\rangle(\hat{\gamma}(G), \hat{\beta}(G))]$ where the expectation is over a synthetic Erdős–Rényi training distribution. The surrogate loss is the directly simulated QAOA objective, making the GNN end-to-end differentiable through the statevector simulator.
- **Caveat:** results here use an untrained model unless `model.pt` is present. Meaningful quality comparisons require training on $\geq 500$ graphs with `src/train.py`.